In [0]:
dbutils.widgets.text("catalog", "policybench_dev")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Project 4 - Databricks Streaming Silver Layer
# MAGIC
# MAGIC Cleans `bronze_telematics_stream` (the Event Hub-sourced live feed) using the
# MAGIC exact same `clean_telematics_silver()` logic already validated on the batch leg
# MAGIC (`nb_silver_databricks.py`) - deliberately reused, not reimplemented, since the
# MAGIC output schema is identical (device_id, timestamp, PID, value, alarm_class) even
# MAGIC though the source differs (Event Hub vs. Azure SQL). Same dedupe rule, same
# MAGIC alarm_class validity check, same per-PID 4-std-dev outlier flag.
# MAGIC
# MAGIC This is a plain batch read/transform/write, not itself a streaming job - it
# MAGIC reprocesses whatever `nb_streaming_bronze_databricks` has landed so far. Run it
# MAGIC after each bronze streaming run (or on a schedule, if you want the demo to show a
# MAGIC bronze -> silver lag), not continuously.
# MAGIC
# MAGIC Run after `nb_streaming_bronze_databricks` (needs `bronze_telematics_stream`).

# COMMAND ----------

from pyspark.sql import DataFrame, functions as F
from datetime import datetime, timezone


def log_pipeline_run(spark, platform, layer, start_dt, end_dt):
    duration = round((end_dt - start_dt).total_seconds(), 2)
    log_row = spark.createDataFrame([{
        "platform": platform, "layer": layer,
        "start_ts": start_dt, "end_ts": end_dt, "duration_seconds": duration,
    }])
    log_row.write.format("delta").mode("append").saveAsTable("pipeline_run_log")
    print(f"[{platform}/{layer}] duration: {duration}s")

# COMMAND ----------


def clean_telematics_silver(df: DataFrame) -> DataFrame:
    # dedupe - full row match only (device_id, timestamp, PID, value, alarm_class) -
    # same rule as batch; here it also absorbs any Event Hub at-least-once redelivery,
    # not just true source retransmissions
    df = df.dropDuplicates(["device_id", "timestamp", "PID", "value", "alarm_class"])

    # alarm_class must be 0-4 per the documented schema - null out anything else,
    # keep the row, the value reading itself is still usable
    df = df.withColumn(
        "alarm_class",
        F.when((F.col("alarm_class") >= 0) & (F.col("alarm_class") <= 4), F.col("alarm_class")).otherwise(None)
    )

    # per-PID outlier flag (not dropped) - beyond 4 std devs of that PID's own mean,
    # computed fresh over whatever's in this stream batch (not against the batch
    # leg's silver_telematics_events stats - streaming and batch stay independently
    # grounded past the shared producer-side generation stats)
    pid_stats = df.groupBy("PID").agg(F.mean("value").alias("pid_mean"), F.stddev("value").alias("pid_std"))
    pid_stats = pid_stats.fillna({"pid_std": 0.01})
    df = df.join(pid_stats, on="PID", how="left")
    df = df.withColumn("VALUE_OUTLIER", F.abs(F.col("value") - F.col("pid_mean")) > (4 * F.col("pid_std")))
    df = df.drop("pid_mean", "pid_std")

    return df

# COMMAND ----------

start_dt = datetime.now(timezone.utc)

bronze_stream = spark.read.table("bronze_telematics_stream")

silver_stream = clean_telematics_silver(
    bronze_stream.select("device_id", "timestamp", "PID", "value", "alarm_class")
)
silver_stream.write.format("delta").mode("overwrite").saveAsTable("silver_telematics_stream")

end_dt = datetime.now(timezone.utc)
log_pipeline_run(spark, "Databricks", "streaming_silver", start_dt, end_dt)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Sanity check
# MAGIC Row count should be at or just under `bronze_telematics_stream`'s count (dedupe
# MAGIC only drops true full-row duplicates, which should be rare for a fresh simulated
# MAGIC feed - if you see a big drop, check whether the producer notebook was run twice
# MAGIC without changing `DEVICE_PREFIX`/timestamps, which would generate a lot of
# MAGIC coincidental duplicates).

# COMMAND ----------

bronze_stream_check = spark.read.table("bronze_telematics_stream")
silver_stream_check = spark.read.table("silver_telematics_stream")

print("bronze_telematics_stream rows:", bronze_stream_check.count())
print("silver_telematics_stream rows:", silver_stream_check.count())
print("silver_telematics_stream outlier rows:", silver_stream_check.filter(F.col("VALUE_OUTLIER")).count())
print("silver_telematics_stream null alarm_class rows:", silver_stream_check.filter(F.col("alarm_class").isNull()).count())